![image.png](https://i.imgur.com/a3uAqnb.png)

# Transformer-XL: Segment-Level Recurrence on WikiText-2

- **Dataset**: [WikiText-2](https://huggingface.co/datasets/Salesforce/wikitext) (raw) from Hugging Face: 720 Wikipedia articles, about 13 million bytes
- **Task**: byte-level language modelling, predicting the next byte of an article
- **Model**: a 6-layer Transformer-XL (about 5.3M parameters), trained from scratch without and with its memory
- **Runtime**: Colab T4 GPU, about 16 minutes of training

In this lab, we will:

- measure context fragmentation as the loss at each position of a segment
- implement segment-level recurrence: a per-layer cache of the previous segment
- implement relative position scores, which keep cached positions consistent
- measure how the test loss changes with the memory length at evaluation
- compare cached evaluation with sliding-window evaluation

## What is segment-level recurrence?

The cost of attention grows with the square of the input length, so a Transformer language model is trained on segments of $s$ tokens cut from a long text, each processed on its own. The first tokens of a segment are then predicted with almost no context: Dai et al. (2019) call this **context fragmentation**.

**Segment-level recurrence** reads the segments in order and caches, for every layer, the states the previous segment produced. Layer $\ell$ of segment $\tau$ attends over the cache and over its own states:

$$\tilde{H}^{\ell-1}_{\tau} = \big[\,\mathrm{SG}(H^{\ell-1}_{\tau-1}) \circ H^{\ell-1}_{\tau}\,\big], \qquad H^{\ell}_{\tau} = \mathrm{Layer}\big(H^{\ell-1}_{\tau} W_q,\; \tilde{H}^{\ell-1}_{\tau} W_k,\; \tilde{H}^{\ell-1}_{\tau} W_v\big)$$

Where:
- $H^{\ell}_{\tau} \in \mathbb{R}^{s \times d}$: the layer-$\ell$ states of segment $\tau$ ($d$: model width; $H^{0}$: the token embeddings)
- $\mathrm{SG}$: the stop-gradient; the cache is a constant in the backward pass, so no gradient enters an earlier segment
- $\circ$: concatenation along the sequence. Here the cache is one segment; in general each layer keeps its last $M$ states, the **memory length**, so the $s$ queries meet $M + s$ keys and values
- $W_q, W_k, W_v$: the query, key and value projections; $\mathrm{Layer}$: causal attention, then the feed-forward block

With $M = s$, each layer looks one segment further back than the layer below: $L$ layers reach about $L \cdot s$ tokens for one pass per segment.

Paper: [Dai et al., 2019. Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context](https://arxiv.org/abs/1901.02860)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. Another Colab GPU also works if no T4 is free.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

hf_logging.set_verbosity_error()   # the dataset is public: hide the notice that no Hugging Face token is set

In [ ]:
SEED = 42
N_TRAIN_TOKENS = 11_000_000   # training bytes to use; the split has about 10.9 million, so this is all of it
N_VAL_TOKENS = 1_200_000      # validation bytes, scored after every epoch (all of the split)
N_TEST_TOKENS = 1_300_000     # test bytes for the final comparisons (all of the split)
N_TEST_SLIDE = 65_536         # test bytes for the slow sliding-window comparison of section 7
SEG_LEN = 128                 # s: bytes per segment
MEM_LEN = 128                 # M: cached states per layer when the memory is on
MEM_SWEEP = [0, 16, 32, 64, 128, 256, 512, 1024]   # memory lengths tried at evaluation in section 6
N_LAYERS = 6                  # L
D_MODEL = 256                 # d
N_HEADS = 4                   # attention heads, each of width D_MODEL / N_HEADS = 64
D_FF = 1024                   # hidden width of the feed-forward block
DROPOUT = 0.1
BATCH_SIZE = 32               # parallel streams of text, each read one segment per step
EPOCHS = 4
LR = 1e-3                     # peak learning rate
WARMUP_STEPS = 200
GRAD_CLIP = 0.5
VOCAB = 256                   # one token per byte value

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

## 1. Dataset

- **Source**: [WikiText](https://huggingface.co/datasets/Salesforce/wikitext) (Merity et al., 2017), built from Wikipedia articles rated Good or Featured; the raw version, meant for character-level models, keeps the rare words that the word-level version replaces with `<unk>`
- **Content**: WikiText-2 has 600 training, 60 validation and 60 test articles (Merity et al., 2017, Table 1)
- **Licence**: CC BY-SA 3.0 and GFDL in the card's metadata (its licensing section says CC BY-SA 4.0)
- **What we use**: all three splits, each joined into one long text and encoded as UTF-8 bytes

In [ ]:
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(wikitext)


def to_bytes(split, n_bytes):
    """Joins the lines of a split into one text; returns its first n_bytes UTF-8 bytes as a LongTensor."""
    text = "".join(wikitext[split]["text"])
    return torch.frombuffer(bytearray(text.encode("utf-8")), dtype=torch.uint8)[:n_bytes].long()


def show(byte_values):
    """Decodes bytes for printing; a multi-byte character cut at the edge becomes a replacement mark."""
    return bytes(byte_values.tolist()).decode("utf-8", errors="replace")


train_data = to_bytes("train", N_TRAIN_TOKENS)
val_data = to_bytes("validation", N_VAL_TOKENS)
test_data = to_bytes("test", N_TEST_TOKENS)
print(f"bytes: train {len(train_data):,} | validation {len(val_data):,} | test {len(test_data):,}")

Each row is one line of an article. Headings look like ` = Title = `; punctuation is split off with spaces and an inner hyphen is written ` @-@ `. The start of the training text:

In [ ]:
print(show(train_data[:700]))

We measure every model in **bits per byte** (bpb), the average cross-entropy of the next byte in bits; in this almost all-ASCII text, that is close to the bits per character (bpc) of character-level papers. A model that ignores context can do no better than the byte frequencies, here counted on the training text; a uniform guess over 256 values costs 8 bpb.

In [ ]:
counts = torch.bincount(train_data, minlength=VOCAB).double() + 1   # add one, so that no byte value has probability 0
unigram_bpb = -counts.div(counts.sum()).log2()[test_data].mean().item()
print(f"byte values seen in training: {(counts > 1).sum().item()} of {VOCAB}")
print(f"byte frequencies alone: {unigram_bpb:.3f} bits per byte on the test text")

### Streams and segments

The memory helps only if segment $\tau + 1$ continues the text of segment $\tau$. We cut each split into `BATCH_SIZE` contiguous **streams**, one per row: step $\tau$ reads bytes $\tau s$ to $\tau s + s - 1$ of every stream as input and the bytes one further on as targets, so row $b$ of each step continues row $b$ of the step before.

In [ ]:
def batchify(data, n_streams):
    """Cuts a 1-D byte tensor into n_streams contiguous rows of n_segments * SEG_LEN + 1 bytes, on DEVICE."""
    n_segments = (len(data) // n_streams - 1) // SEG_LEN
    if n_segments < 1:
        raise ValueError(f"{len(data):,} bytes are too few for {n_streams} streams of one {SEG_LEN}-byte segment")
    length = n_segments * SEG_LEN + 1          # the extra byte is the target of the last input byte
    return data[:n_streams * length].view(n_streams, length).to(DEVICE)


def get_segment(streams, t):
    """Inputs and targets of segment t of every stream: two (n_streams, SEG_LEN) tensors."""
    start = t * SEG_LEN
    return streams[:, start:start + SEG_LEN], streams[:, start + 1:start + SEG_LEN + 1]


def n_segments(streams):
    return (streams.size(1) - 1) // SEG_LEN


train_streams = batchify(train_data, BATCH_SIZE)
val_streams = batchify(val_data, BATCH_SIZE)
test_streams = batchify(test_data, BATCH_SIZE)
for name, streams in [("train", train_streams), ("validation", val_streams), ("test", test_streams)]:
    print(f"{name:<10} streams {tuple(streams.shape)}: {n_segments(streams):,} segments per stream")

for t in range(2):
    x, y = get_segment(train_streams, t)
    print(f"\nsegment {t}, stream 1, input:\n{show(x[1])!r}")

## 2. Model

### Relative positions

With absolute position embeddings, token $j$ of every segment would get the same position, so the model could not tell a cached state from a current one. Transformer-XL instead puts the distance $i - j$ between query $i$ and key $j$ into every attention score:

$$A_{i,j} = \underbrace{q_i^{\top} k_j}_{(a)} + \underbrace{q_i^{\top} W_{k,R}\, r_{i-j}}_{(b)} + \underbrace{u^{\top} k_j}_{(c)} + \underbrace{v^{\top} W_{k,R}\, r_{i-j}}_{(d)}$$

Where:
- $q_i$, $k_j$: the query and the key, computed from content alone
- $r_{i-j}$: a fixed sinusoidal encoding of the distance; $W_{k,R}$: its learned key projection
- $u$, $v$: learned vectors shared by all positions; (a) is content addressing, (b) a position term that depends on the query's content, (c) a global content bias, (d) a global position bias

The code computes (a) + (c) as $(q_i + u)^{\top} k_j$ and (b) + (d) as $(q_i + v)^{\top} W_{k,R}\, r_t$ for every distance $t$, then picks for each query-key pair the entry of its distance.

In [ ]:
class RelativeAttention(nn.Module):
    """Causal multi-head attention over [memory; segment] with Transformer-XL's relative position scores."""

    def __init__(self):
        super().__init__()
        self.d_head = D_MODEL // N_HEADS
        self.w_q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.w_kv = nn.Linear(D_MODEL, 2 * D_MODEL, bias=False)             # W_k and W_v in one matrix
        self.w_kr = nn.Linear(D_MODEL, D_MODEL, bias=False)             # W_{k,R}, applied to the distance encodings
        self.u = nn.Parameter(torch.zeros(N_HEADS, self.d_head))        # global content bias, term (c)
        self.v = nn.Parameter(torch.zeros(N_HEADS, self.d_head))        # global position bias, term (d)
        self.w_o = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x, context, r):
        # x: (B, s, d) the segment; context: (B, M + s, d) memory then segment; r: (M + s, d) encodings of distances 0, 1, ...
        B, s, _ = x.shape
        n_keys = context.size(1)
        M = n_keys - s
        q = self.w_q(x).view(B, s, N_HEADS, self.d_head)                                     # (B, s, h, d_h)
        k, values = self.w_kv(context).view(B, n_keys, 2, N_HEADS, self.d_head).unbind(2)   # each (B, M + s, h, d_h)
        r_k = self.w_kr(r).view(n_keys, N_HEADS, self.d_head)                                # row t: distance t
        content = torch.einsum("bihd,bjhd->bhij", q + self.u, k)           # (a) + (c): (B, h, s, M + s)
        position = torch.einsum("bihd,thd->bhit", q + self.v, r_k)         # (b) + (d) for each distance t
        # query i sits at position M + i of the context, so its distance to key j is M + i - j
        dist = M + torch.arange(s, device=x.device)[:, None] - torch.arange(n_keys, device=x.device)[None, :]
        position = position.gather(-1, dist.clamp(min=0).expand(B, N_HEADS, s, n_keys))
        scores = (content + position) / math.sqrt(self.d_head)
        scores = scores.masked_fill(dist < 0, float("-inf"))              # causal: a negative distance is a future key
        attn = self.dropout(scores.float().softmax(dim=-1).to(values.dtype))
        out = torch.einsum("bhij,bjhd->bihd", attn, values).reshape(B, s, D_MODEL)
        return self.w_o(out)

### Layers and memory

Each block attends over $\tilde{H}^{\ell-1}_{\tau}$: its cache, then the segment. The model returns the logits and every layer's input states $H^{0}_{\tau}, \dots, H^{L-1}_{\tau}$; `update_memory` appends these to the caches, keeps the last $M$ per layer and detaches them from the graph, which is the stop-gradient $\mathrm{SG}$. With $M = 0$ the cache stays empty and every segment is read on its own.

In [ ]:
class XLBlock(nn.Module):
    """Pre-norm Transformer block; its attention also reads the cached states of earlier segments."""

    def __init__(self):
        super().__init__()
        self.norm_attn = nn.LayerNorm(D_MODEL)
        self.attn = RelativeAttention()
        self.norm_ff = nn.LayerNorm(D_MODEL)
        self.ff = nn.Sequential(nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(D_FF, D_MODEL))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, memory, r):
        # h: (B, s, d) this layer's input; memory: (B, M, d) its cached input of earlier segments
        context = self.norm_attn(torch.cat([memory, h], dim=1))        # normalised tilde-H: (B, M + s, d)
        h = h + self.dropout(self.attn(context[:, memory.size(1):], context, r))
        return h + self.dropout(self.ff(self.norm_ff(h)))


class TransformerXL(nn.Module):
    """Byte-level language model; forward returns the logits and every layer's input states."""

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D_MODEL)
        self.blocks = nn.ModuleList([XLBlock() for _ in range(N_LAYERS)])
        self.norm_out = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB)
        self.dropout = nn.Dropout(DROPOUT)
        inv_freq = 1.0 / 10_000 ** (torch.arange(0, D_MODEL, 2).float() / D_MODEL)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def distance_encoding(self, n):
        """Sinusoidal encodings r_t of the distances t = 0, ..., n - 1: (n, d)."""
        angles = torch.arange(n, device=self.inv_freq.device).float()[:, None] * self.inv_freq[None, :]
        return torch.cat([angles.sin(), angles.cos()], dim=-1)

    def forward(self, x, memory):
        # x: (B, s) byte values; memory: one (B, M, d) tensor per layer
        h = self.dropout(self.embed(x))                                    # H^0: (B, s, d)
        r = self.dropout(self.distance_encoding(memory[0].size(1) + x.size(1)))
        hiddens = []
        for block, mem in zip(self.blocks, memory, strict=True):          # a cache built for another N_LAYERS raises here
            hiddens.append(h)                                              # H^{l-1}, to be cached for the next segment
            h = block(h, mem, r)
        return self.head(self.norm_out(h)), hiddens                        # logits: (B, s, VOCAB)


def empty_memory(n_streams):
    """A cache with no states yet: one (n_streams, 0, d) tensor per layer."""
    return [torch.zeros(n_streams, 0, D_MODEL, device=DEVICE) for _ in range(N_LAYERS)]


def update_memory(memory, hiddens, mem_len):
    """Appends each layer's new input states to its cache, keeps the last mem_len, and stops the gradient."""
    new_memory = []
    for mem, h in zip(memory, hiddens):
        both = torch.cat([mem, h.detach()], dim=1)                         # (B, old M + s, d)
        new_memory.append(both[:, max(0, both.size(1) - mem_len):])
    return new_memory

One model, fed the first three training segments: the shapes, and the cache filling up.

In [ ]:
model = TransformerXL().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

memory = empty_memory(BATCH_SIZE)
with torch.no_grad():
    for t in range(3):
        x, _ = get_segment(train_streams, t)
        logits, hiddens = model(x, memory)
        print(f"segment {t}: input {tuple(x.shape)}, memory read {tuple(memory[0].shape)}, logits {tuple(logits.shape)}")
        memory = update_memory(memory, hiddens, MEM_LEN)
del model, memory

## 3. Training

Both models start from the same weights and read the same segments in order; only the memory length differs. The loss is the next-byte cross-entropy in bpb, the natural-log loss divided by $\ln 2$. AdamW's learning rate warms up for `WARMUP_STEPS` steps, then follows a cosine to a tenth of `LR`; gradients are clipped at norm `GRAD_CLIP`.

In [ ]:
def learning_rate(step, total_steps):
    """Linear warm-up, then cosine decay from LR to LR / 10."""
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return LR * (0.1 + 0.45 * (1 + math.cos(math.pi * progress)))


def train_one_epoch(model, optimizer, scaler, mem_len, first_step, total_steps):
    """One pass over the training streams, segment by segment, carrying the memory. Returns the mean bpb."""
    model.train()
    memory = empty_memory(train_streams.size(0))
    total = 0.0
    progress = tqdm(range(n_segments(train_streams)), desc="train", leave=False)
    for t in progress:
        for group in optimizer.param_groups:
            group["lr"] = learning_rate(first_step + t, total_steps)
        x, y = get_segment(train_streams, t)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, hiddens = model(x, memory)                             # (B, s, VOCAB)
            loss = F.cross_entropy(logits.float().reshape(-1, VOCAB), y.reshape(-1))
        memory = update_memory(memory, hiddens, mem_len)                   # detached: no gradient into older segments
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                                      # the scaler is a no-op when disabled
        scaler.unscale_(optimizer)                                         # clip the true gradients, not the scaled ones
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        progress.set_postfix(bpb=f"{loss.item() / math.log(2):.3f}")
    return total / n_segments(train_streams) / math.log(2)


@torch.no_grad()
def byte_losses(model, streams, mem_len):
    """Loss in bits of every predicted byte, reading the segments in order with a memory of mem_len states per layer.
    Returns (n_streams, n_segments * SEG_LEN); column j holds the loss of byte j + 1 of each stream."""
    model.eval()
    memory = empty_memory(streams.size(0))
    losses = []
    for t in range(n_segments(streams)):
        x, y = get_segment(streams, t)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, hiddens = model(x, memory)
        memory = update_memory(memory, hiddens, mem_len)
        losses.append(F.cross_entropy(logits.float().transpose(1, 2), y, reduction="none") / math.log(2))
    return torch.cat(losses, dim=1)


def evaluate(model, streams, mem_len):
    """Mean bits per byte over the streams."""
    return byte_losses(model, streams, mem_len).mean().item()


def train_model(mem_len):
    """Trains a fresh TransformerXL that carries mem_len cached states per layer; returns it and its history."""
    torch.manual_seed(SEED)                                                # the same initial weights for every call
    model = TransformerXL().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    steps_per_epoch = n_segments(train_streams)
    print(f"memory length {mem_len}: {steps_per_epoch:,} optimizer steps per epoch, {EPOCHS} epochs")
    history = {"train_bpb": [], "val_bpb": []}
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_bpb = train_one_epoch(model, optimizer, scaler, mem_len, (epoch - 1) * steps_per_epoch,
                                    EPOCHS * steps_per_epoch)
        val_bpb = evaluate(model, val_streams, mem_len)
        history["train_bpb"].append(train_bpb)
        history["val_bpb"].append(val_bpb)
        print(f"Epoch {epoch}/{EPOCHS}: train {train_bpb:.4f} bpb | val {val_bpb:.4f} bpb | {time.time() - start:.0f}s elapsed")
    return model, history

First the model without memory ($M = 0$), which reads every segment on its own. Expected time: about 7 minutes on a T4.

In [ ]:
vanilla, history_vanilla = train_model(mem_len=0)

Then Transformer-XL: the same model with a memory of `MEM_LEN` states per layer. Each query now scores $M + s$ keys instead of $s$, so a step does more work. Expected time: about 9 minutes on a T4.

In [ ]:
xl, history_xl = train_model(mem_len=MEM_LEN)

## 4. Results

The training bpb is averaged over each epoch with dropout on, so it can sit above the validation bpb.

In [ ]:
NAMES = {"vanilla": "no memory", "xl": f"memory {MEM_LEN}"}
epochs = range(1, EPOCHS + 1)
fig, ax = plt.subplots(figsize=(7, 4))
for key, history, color in [("vanilla", history_vanilla, "tab:blue"), ("xl", history_xl, "tab:orange")]:
    ax.plot(epochs, history["train_bpb"], marker="o", linestyle="--", color=color, label=f"{NAMES[key]}, train")
    ax.plot(epochs, history["val_bpb"], marker="o", color=color, label=f"{NAMES[key]}, validation")
ax.set(xlabel="epoch", ylabel="bits per byte", title="Training curves", xticks=list(epochs))
ax.legend()
plt.show()

Test bits per byte, each model with its training memory length:

In [ ]:
test_losses = {"vanilla": byte_losses(vanilla, test_streams, 0), "xl": byte_losses(xl, test_streams, MEM_LEN)}
print(f"{'byte frequencies alone':<24} {unigram_bpb:.4f} bits per byte")
for key, losses in test_losses.items():
    print(f"{NAMES[key]:<24} {losses.mean().item():.4f} bits per byte")

## 5. Where the Memory Helps: Loss by Position

We average the test loss over all segments at each position inside a segment, 0 to $s - 1$. Without memory, the prediction at position $p$ sees the $p + 1$ bytes of its segment up to $p$; with memory, also the cache.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for key, losses in test_losses.items():
    per_position = losses.view(losses.size(0), -1, SEG_LEN).mean(dim=(0, 1)).cpu()
    ax.plot(range(SEG_LEN), per_position, label=NAMES[key])
    print(f"{NAMES[key]:<10} position 0: {per_position[0]:.3f} | positions 0-15: {per_position[:16].mean():.3f} | "
          f"positions {SEG_LEN // 2}-{SEG_LEN - 1}: {per_position[SEG_LEN // 2:].mean():.3f} bits per byte")
ax.set(xlabel="position inside the segment", ylabel="test bits per byte", title="Loss by position in the segment")
ax.legend()
plt.show()

**What to notice**

Without memory, the prediction at position 0 sees a single byte and its loss is far above the average; the loss falls steeply over the first positions. With memory the curve is flat from position 0. Much of the gap between the models comes from the start of the segments, but the memory model stays lower to the end: after the first segment of each stream, it sees at least `MEM_LEN` more bytes at every position.

## 6. How Much Memory Does the Model Use?

Relative position scores are defined for any distance, so a trained model can run with any memory length. We score the test text with each length in `MEM_SWEEP`. Expected time: about 2 minutes on a T4.

In [ ]:
sweep = {"vanilla": [], "xl": []}
for mem_len in tqdm(MEM_SWEEP, desc="memory lengths"):
    sweep["vanilla"].append(evaluate(vanilla, test_streams, mem_len))
    sweep["xl"].append(evaluate(xl, test_streams, mem_len))

print("memory length M:   " + "".join(f"{m:>8}" for m in MEM_SWEEP))
for key, values in sweep.items():
    print(f"{NAMES[key]:<18} " + "".join(f"{v:8.4f}" for v in values))

fig, ax = plt.subplots(figsize=(8, 4))
x_values = [max(m, 8) for m in MEM_SWEEP]                                  # M = 0 drawn at 8 on the log axis
for key in sweep:
    ax.plot(x_values, sweep[key], marker="o", label=f"trained with {NAMES[key]}")
ax.axvline(MEM_LEN, color="gray", linestyle="--", linewidth=1, label=f"training memory, M = {MEM_LEN}")
ax.set_xscale("log", base=2)
ax.set_xticks(x_values, ["0"] + [str(m) for m in MEM_SWEEP[1:]])
ax.set(xlabel="memory length M at evaluation (states per layer)", ylabel="test bits per byte",
       title="Test loss against the evaluation memory")
ax.legend()
plt.show()

**What to notice**

Both models gain from a cache, even the one trained without it, mostly from the first few dozen cached states per layer: a cached state of a higher layer already summarises its own segment, so with $0 < M \leq s$ the predictions reach $(L - 1)s + M$ bytes before their segment. The memory model gains far more, and without its memory ($M = 0$) it does worse than the model trained without one. Beyond `MEM_LEN`, the memory model's curve flattens, then rises again.

## 7. The Cost of Recomputing the Context

At evaluation time, the model without memory can avoid fragmentation by sliding its window one byte at a time: each byte is predicted from exactly the $s$ bytes before it, at the cost of a forward pass over $s$ positions per byte. From the first `N_TEST_SLIDE` test bytes, we score the same bytes three ways and time each. Expected time: under a minute on a T4.

In [ ]:
@torch.no_grad()
def sliding_window_losses(model, streams):
    """Loss in bits of bytes SEG_LEN onward of each stream, each predicted from exactly the SEG_LEN bytes before it:
    one forward pass per byte, without memory. Returns (n_streams, length - SEG_LEN)."""
    model.eval()
    no_memory = empty_memory(streams.size(0))
    losses = []
    for p in tqdm(range(SEG_LEN, streams.size(1)), desc="sliding window", leave=False):
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, _ = model(streams[:, p - SEG_LEN:p], no_memory)         # only the last position is used
        losses.append(F.cross_entropy(logits[:, -1].float(), streams[:, p], reduction="none") / math.log(2))
    return torch.stack(losses, dim=1)


def timed(fn, *args):
    """Runs fn(*args) and returns its result and the wall time in seconds."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    result = fn(*args)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return result, time.perf_counter() - start


slide_streams = batchify(test_data[:N_TEST_SLIDE], BATCH_SIZE)
fragmented, t_fragmented = timed(byte_losses, vanilla, slide_streams, 0)
sliding, t_sliding = timed(sliding_window_losses, vanilla, slide_streams)
cached, t_cached = timed(byte_losses, xl, slide_streams, MEM_LEN)
# column j of byte_losses is byte j + 1: keep bytes SEG_LEN onward, the ones the sliding window predicts
fragmented, cached = fragmented[:, SEG_LEN - 1:], cached[:, SEG_LEN - 1:]

rows = [("no memory, one pass per segment", n_segments(slide_streams), t_fragmented, fragmented),
        ("no memory, sliding-window evaluation", sliding.size(1), t_sliding, sliding),
        (f"memory {MEM_LEN}, one pass per segment", n_segments(slide_streams), t_cached, cached)]
print(f"{sliding.numel():,} test bytes scored by each method, {slide_streams.size(0)} streams at a time")
print(f"{'method':<38} {'passes':>7} {'seconds':>8} {'bits per byte':>14}")
for name, passes, seconds, losses in rows:
    print(f"{name:<38} {passes:>7,} {seconds:>8.2f} {losses.mean().item():>14.4f}")

**What to notice**

Sliding-window evaluation lowers the loss of the model without memory but takes one forward pass per byte instead of one per segment. The cached model reaches a lower loss in a small fraction of the time. Dai et al. (2019, Table 9) measured evaluation up to 1,874 times faster per token than the fixed-context Transformer of Al-Rfou et al. with a 3,800-token sliding window.

## 8. Save and Reload

We save the Transformer-XL weights, reload them and check the test loss.

In [ ]:
torch.save(xl.state_dict(), "transformer_xl_wikitext2.pth")

reloaded = TransformerXL().to(DEVICE)
reloaded.load_state_dict(torch.load("transformer_xl_wikitext2.pth", map_location=DEVICE))
print(f"reloaded model, memory {MEM_LEN}: {evaluate(reloaded, test_streams, MEM_LEN):.4f} test bits per byte "
      f"(before saving: {test_losses['xl'].mean().item():.4f})")

## Summary

- Independent segments leave the first bytes of each segment with little context: context fragmentation.
- Segment-level recurrence caches each layer's input states for the next segment; the stop-gradient keeps the backward pass inside the current segment.
- Relative position scores keep cached states at consistent distances, so the memory length can change at evaluation.
- A cache helps most when the model was trained with it.
- Reusing the cache is far cheaper than recomputing the context for every byte.

## Exercises

Copy the printed results first: a rerun overwrites them. Start each exercise from the original constants and code.

1. **Absolute positions.** In `RelativeAttention.forward`, set `scores = content / math.sqrt(self.d_head)`; in `TransformerXL`, add a learned `nn.Embedding(SEG_LEN, D_MODEL)` of the positions 0 to $s - 1$ to the byte embeddings. Re-run the section 2 cells, the memory model's training cell and sections 4 and 5: what happens at the start of each segment, and why?
2. **A longer training memory.** Set `MEM_LEN = 256` and run all cells again. Does the test loss improve, and where does the section 6 curve flatten? How does the epoch time change?
3. **Shorter segments.** Set `SEG_LEN = 64`, `MEM_LEN = 64` and `BATCH_SIZE = 64` (the same bytes per step) and run all cells again. Whose test loss rises more, and why?
4. **Reach through depth.** Set `N_LAYERS = 2` and run all cells again. With $M = s$, the reach $L \cdot s$ shrinks from 768 to 256 bytes: does the gap between the two models shrink?

## Further Reading

- [Dai et al., 2019. Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context](https://arxiv.org/abs/1901.02860)
- [Al-Rfou et al., 2019. Character-Level Language Modeling with Deeper Self-Attention](https://arxiv.org/abs/1808.04444)
- [Rae et al., 2020. Compressive Transformers for Long-Range Sequence Modelling](https://arxiv.org/abs/1911.05507)
- [Bulatov et al., 2022. Recurrent Memory Transformer](https://arxiv.org/abs/2207.06881)
- [Merity et al., 2017. Pointer Sentinel Mixture Models](https://arxiv.org/abs/1609.07843)

### Contributed by: Ali Habibullah